# InHire API Scanner

Notebook para descobrir endpoints públicos da plataforma InHire.

Ele testa múltiplos endpoints possíveis e identifica:

- status HTTP
- se retorna JSON
- tamanho da resposta
- preview do conteúdo

O resultado final é salvo em CSV.

In [ ]:
import requests
import pandas as pd
from pprint import pprint

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*"
}

## Empresas para teste

In [ ]:
EMPRESAS_TESTE = [
    "sympla",
    "magalu",
    "olist",
    "cora",
    "contabilizei"
]

## Possíveis endpoints da InHire

In [ ]:
ENDPOINTS = [

    "https://{empresa}.inhire.app/api/jobs",
    "https://{empresa}.inhire.app/api/vagas",
    "https://{empresa}.inhire.app/api/positions",
    "https://{empresa}.inhire.app/api/opportunities",
    "https://{empresa}.inhire.app/api/list",

    "https://{empresa}.inhire.app/api/v1/jobs",
    "https://{empresa}.inhire.app/api/v1/vagas",
    "https://{empresa}.inhire.app/api/v1/positions",

    "https://api.inhire.app/jobs",
    "https://api.inhire.app/vagas",
    "https://api.inhire.app/v1/jobs",
    "https://api.inhire.app/v1/vagas",

    "https://{empresa}.inhire.app/graphql",
    "https://api.inhire.app/graphql",

    "https://{empresa}.inhire.app/jobs.json",
    "https://{empresa}.inhire.app/vagas.json",

    "https://api.inhire.app/search/jobs",
    "https://api.inhire.app/search/vagas"
]

## Função para testar endpoint

In [ ]:
def testar_endpoint(url):

    try:

        r = requests.get(url, headers=HEADERS, timeout=15)

        content_type = r.headers.get("content-type", "")

        if "json" in content_type:
            try:
                data = r.json()
                tamanho = len(str(data))

                return {
                    "url": url,
                    "status": r.status_code,
                    "json": True,
                    "size": tamanho,
                    "preview": str(data)[:300]
                }

            except:
                pass

        return {
            "url": url,
            "status": r.status_code,
            "json": False,
            "size": len(r.text),
            "preview": r.text[:200]
        }

    except Exception as e:

        return {
            "url": url,
            "status": "erro",
            "json": False,
            "size": 0,
            "preview": str(e)
        }

## Scanner de APIs

In [ ]:
resultados = []

for empresa in EMPRESAS_TESTE:

    print("\n🔎 Empresa:", empresa)

    for ep in ENDPOINTS:

        url = ep.format(empresa=empresa)

        print("Testando:", url)

        res = testar_endpoint(url)

        res["empresa"] = empresa

        resultados.append(res)

## Resultados

In [ ]:
df = pd.DataFrame(resultados)

df

## Filtrar endpoints promissores

In [ ]:
df_promissor = df[(df["status"] == 200) & (df["size"] > 500)]

df_promissor

## Salvar relatório

In [ ]:
df.to_csv("inhire_api_scan.csv", index=False)

print("Relatório salvo como inhire_api_scan.csv")